In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



sns.set_theme(style="whitegrid")

# Reload and filter again
df = pd.read_csv(
    r'C:\Users\pench\Documents\credit-risk-project\data\raw\accepted_2007_to_2018Q4.csv',
    low_memory=False
)
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])]

selected_cols = [
    'loan_amnt', 'int_rate', 'installment', 'grade',
    'emp_length', 'home_ownership', 'annual_inc', 'purpose',
    'dti', 'fico_range_low', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'loan_status'
]
df = df[selected_cols].copy()

print(f" Data loaded! Shape: {df.shape}")

 Data loaded! Shape: (1345310, 16)


In [6]:
# Check current format
print("Before:", df['int_rate'].head())

df['int_rate'] = df['int_rate'].astype(str).str.replace('%','').str.strip().astype(float)

print("After:", df['int_rate'].head())
print("int_rate fixed!")



Before: 0    13.99
1    11.99
2    10.78
4    22.45
5    13.44
Name: int_rate, dtype: float64
After: 0    13.99
1    11.99
2    10.78
4    22.45
5    13.44
Name: int_rate, dtype: float64
int_rate fixed!


In [7]:
print("Unique values:", df['emp_length'].unique())

emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10
}
df['emp_length'] = df['emp_length'].map(emp_map)

print("\nAfter mapping:")
print(df['emp_length'].value_counts(dropna=False))
print("emp_length fixed!")

Unique values: <StringArray>
['10+ years',   '3 years',   '4 years',   '6 years',   '7 years',   '8 years',
   '2 years',   '5 years',   '9 years',  '< 1 year',    '1 year',         nan]
Length: 12, dtype: str

After mapping:
emp_length
10.0    442199
2.0     121743
0.0     108061
3.0     107597
1.0      88494
5.0      84154
4.0      80556
NaN      78511
6.0      62733
8.0      60701
7.0      59624
9.0      50937
Name: count, dtype: int64
emp_length fixed!


In [8]:
# Check missing before
print("Missing values BEFORE:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill numeric nulls with median
df['revol_util'] = df['revol_util'].fillna(df['revol_util'].median())
df['emp_length'] = df['emp_length'].fillna(-1)  # -1 means unknown

# Drop rows where critical columns are null
df.dropna(subset=['annual_inc', 'dti', 'fico_range_low'], inplace=True)

print("\nMissing values AFTER:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\n Cleaning done! Shape: {df.shape}")

Missing values BEFORE:
emp_length    78511
dti             374
revol_util      857
dtype: int64

Missing values AFTER:
Series([], dtype: int64)

 Cleaning done! Shape: (1344936, 16)


In [9]:
# 0 = Fully Paid (Good), 1 = Charged Off (Default)
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)
df.drop('loan_status', axis=1, inplace=True)

print("Target variable distribution:")
print(df['default'].value_counts())
print(f"\n Target encoded! 0=Good, 1=Default")

Target variable distribution:
default
0    1076448
1     268488
Name: count, dtype: int64

 Target encoded! 0=Good, 1=Default


In [10]:
# Cap annual_inc and revol_bal at 99th percentile
for col in ['annual_inc', 'revol_bal']:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)
    print(f" {col} capped at {cap:,.0f}")

 annual_inc capped at 250,000
 revol_bal capped at 94,554


In [11]:
# Loan to income ratio — how big is the loan vs their income?
df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + 1)

# Monthly burden — what % of monthly income goes to installment?
df['monthly_burden'] = df['installment'] / (df['annual_inc'] / 12 + 1)

print("New features created!")
print(df[['loan_to_income', 'monthly_burden']].describe().round(4))

New features created!
       loan_to_income  monthly_burden
count    1.344936e+06    1.344936e+06
mean     2.188000e-01    8.050000e-02
std      9.810000e-01    2.695000e-01
min      4.000000e-03    6.000000e-04
25%      1.250000e-01    4.650000e-02
50%      2.000000e-01    7.220000e-02
75%      2.909000e-01    1.054000e-01
max      8.000000e+02    2.160824e+02


In [12]:
print("Before encoding shape:", df.shape)

df = pd.get_dummies(
    df,
    columns=['home_ownership', 'purpose', 'grade'],
    drop_first=True
)

print("After encoding shape:", df.shape)
print("Categorical columns encoded!")

Before encoding shape: (1344936, 18)
After encoding shape: (1344936, 39)
Categorical columns encoded!


In [21]:
df.to_csv('../data/processed/cleaned_loans.csv', index=False)

print(" Cleaned data saved!")
print(" Final shape:", df.shape)
print("\nColumn list:")
for col in df.columns:
    print(" -", col)

 Cleaned data saved!
 Final shape: (1344936, 39)

Column list:
 - loan_amnt
 - int_rate
 - installment
 - emp_length
 - annual_inc
 - dti
 - fico_range_low
 - open_acc
 - pub_rec
 - revol_bal
 - revol_util
 - total_acc
 - default
 - loan_to_income
 - monthly_burden
 - home_ownership_MORTGAGE
 - home_ownership_NONE
 - home_ownership_OTHER
 - home_ownership_OWN
 - home_ownership_RENT
 - purpose_credit_card
 - purpose_debt_consolidation
 - purpose_educational
 - purpose_home_improvement
 - purpose_house
 - purpose_major_purchase
 - purpose_medical
 - purpose_moving
 - purpose_other
 - purpose_renewable_energy
 - purpose_small_business
 - purpose_vacation
 - purpose_wedding
 - grade_B
 - grade_C
 - grade_D
 - grade_E
 - grade_F
 - grade_G
